# MLP(1) + MNIST | Adam과 Adam-HD

[Jie et al. (2022), Adaptive hierarchical hyper-gradient descent](https://doi.org/10.1007/s13042-022-01625-4)의 표 1, §5.3, 그림 3을 바탕으로 한다.

| 조건 | Adam | Adam-HD |
|---|---:|---:|
| 초기 학습률 `lr2` | 0.0003 | 0.0003 |
| 하이퍼그래디언트 업데이트 계수 `lr4` | | 1e-7 |
| 배치 크기 | 32 | 32 |
| 학습 길이 | 30 epoch | 30 epoch |
| 반복 | 10회 | 10회 |

MNIST 공식 학습 60,000장을 모두 사용하고 공식 테스트 10,000장으로 매 epoch 평가한다. §5.3과 표 2의 순서에 따라 MLP(1)의 은닉층은 `[100, 100]`으로 해석했다. 고정 학습률 스케줄은 없으며 Adam-HD의 학습률은 하이퍼그래디언트로 갱신한다. 표 1의 `1e-7`을 사용한다. §5.3/표 2에 보고된 별도 `1e-9` 비교 조건과 CAM-HD는 실험에 포함하지 않았다.

AHHD에 세부 설정이 명시되지 않은 부분은 논문이 인용한 [Baydin et al. 참조 구현](https://github.com/gbaydin/hypergradient-descent/tree/020d6080c4cedfbc88d5cdb7a2a53f92b34c2b16)을 따랐다. 은닉층 ReLU, 픽셀을 [0, 1]로 변환한 뒤 평균 0.1307 및 표준편차 0.3081로 정규화, 교차 엔트로피, Adam의 β=(0.9, 0.999)·ε=1e-8·weight decay=0을 사용한다. 초기화는 PyTorch Linear 기본값을 쓰며, 논문에 공개되지 않은 시드는 0–9로 정했다. 각 시드의 두 조건에 동일한 초기 가중치와 배치 순서를 사용한다. 원 논문의 Python 3.5.1 / PyTorch 1.0 / V100 대신 현대 PyTorch에서 실행하므로 표의 정확도와 수치가 일치할 필요는 없다.

## 실행

Python 환경에 `torch`, `torchvision`, `numpy`, `matplotlib`, `networkx`를 설치하고 Jupyter에서 위부터 실행한다. 검증 환경은 Python 3.12, PyTorch 2.7.1, torchvision 0.22.1 CPU다. 예를 들어 CPU용 PyTorch는 `python -m pip install torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cpu`로 설치할 수 있다. 나머지는 `python -m pip install numpy matplotlib networkx jupyterlab`로 설치한다.

기본 설정은 전체 실험이다. 빠른 실행은 `NS_SMOKE=1`로 설정하면 시드 1개, 1 epoch, 학습 8 배치로 제한된다. `NS_RUNS=1`만 설정하면 시드 1개로 30 epoch 전체를 검증할 수 있다. 출력과 데이터는 기본적으로 현재 작업 디렉터리의 상위 경로에 있는 `network-science-runs`에 저장된다. 다른 경로가 필요하면 `NS_ROOT`를 지정한다. 전체 배치 추적 파일 20개의 용량은 비압축 기준 약 2.7 GB다.

In [ ]:
import csv
import json
import math
import os
import tempfile
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 32
LR2 = 3e-4
LR4 = 1e-7
BETAS = (0.9, 0.999)
EPS = 1e-8
HIDDEN_SIZE = 100
SMOKE = os.environ.get("NS_SMOKE") == "1"
EPOCHS = 1 if SMOKE else 30
SEEDS = (0,) if SMOKE else tuple(range(int(os.environ.get("NS_RUNS", "10"))))
assert SEEDS, "NS_RUNS는 1 이상이어야 합니다."
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

ROOT = Path(os.environ.get("NS_ROOT", Path.cwd().parent / "network-science-runs"))
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT = Path(tempfile.mkdtemp(prefix="smoke-" if SMOKE else "mlp1-", dir=ROOT))

NODE_IDS = [f"h{layer}:{unit:03d}" for layer in (1, 2) for unit in range(HIDDEN_SIZE)]
METHODS = {"Adam": 0.0, "Adam-HD": LR4}

In [ ]:
class ResearchMNIST(torchvision.datasets.MNIST):
    # 동일한 공식 IDX 파일의 미러. torchvision이 압축 파일의 MD5를 검사한다.
    mirrors = ["https://raw.githubusercontent.com/fgnt/mnist/master/",
               *torchvision.datasets.MNIST.mirrors]


def load_mnist(train):
    dataset = ResearchMNIST(ROOT / "data", train=train, download=True)
    images = dataset.data.flatten(1).float().div_(255).sub_(0.1307).div_(0.3081)
    return TensorDataset(images, dataset.targets)


train_data = load_mnist(train=True)
test_data = load_mnist(train=False)
assert (len(train_data), len(test_data)) == (60_000, 10_000)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)


class MLP1(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(28 * 28, HIDDEN_SIZE)
        self.hidden2 = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.output = nn.Linear(HIDDEN_SIZE, 10)

    def forward(self, images):
        activations = self.hidden1(images).relu()
        activations = self.hidden2(activations).relu()
        return self.output(activations)

## Adam과 Adam-HD 업데이트

PyTorch 1.0 시기의 Adam 표기법에 맞춰 모멘트 분모를 `sqrt(v) + eps`로 두고, 스텝 크기에 bias correction을 적용한다. Adam-HD는 현재 그래디언트와 직전 Adam 방향의 내적에 `lr4`를 곱해 공유 학습률을 갱신한다. AHHD §5에서 인용한 Baydin 공개 구현의 동작에 맞춰 파라미터 텐서를 순회하며 공유 학습률을 더하고, 각 텐서의 스텝에 그 시점의 값을 사용한다. 배치 로그에는 마지막 텐서까지 업데이트한 시점의 공유 학습률을 기록한다.

In [ ]:
class PaperAdam(torch.optim.Optimizer):
    def __init__(self, params, hypergrad_lr):
        super().__init__(params, dict(lr=LR2, hypergrad_lr=hypergrad_lr))

    @torch.no_grad()
    def step(self):
        beta1, beta2 = BETAS
        for group in self.param_groups:
            for parameter in group["params"]:
                grad = parameter.grad
                if grad is None:
                    continue

                state = self.state[parameter]
                if not state:
                    state.update(step=0, first=torch.zeros_like(parameter),
                                 second=torch.zeros_like(parameter))
                previous_step = state["step"]
                first, second = state["first"], state["second"]
                if group["hypergrad_lr"] and previous_step:
                    previous_scale = math.sqrt(1 - beta2 ** previous_step) / (
                        1 - beta1 ** previous_step
                    )
                    previous_direction = first / (second.sqrt() + EPS)
                    inner_product = torch.dot(grad.flatten(), previous_direction.flatten())
                    group["lr"] += group["hypergrad_lr"] * inner_product.item() * previous_scale

                first.mul_(beta1).add_(grad, alpha=1 - beta1)
                second.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                step = previous_step + 1
                state["step"] = step
                scale = math.sqrt(1 - beta2 ** step) / (1 - beta1 ** step)
                parameter.addcdiv_(first, second.sqrt().add_(EPS), value=-group["lr"] * scale)

## 배치·epoch별 뉴런 추적

노드는 두 은닉층의 뉴런 200개이며 노드 ID는 시드가 동일한 두 조건 간에 초기화 시점부터 대응한다. 가중치 norm은 각 뉴런의 입력 가중치 벡터의 L2 norm으로 정의하며 bias를 포함하지 않는다. 배치별로 업데이트 직전 그래디언트 norm과 실제 가중치 변화량 norm도 저장한다. `step=0`은 초기 가중치다. `epoch_steps`는 초기 상태 및 각 epoch 마지막 업데이트에 해당하는 배치 행 인덱스다. 그래디언트와 변화량의 epoch 스냅샷에는 해당 epoch의 마지막 배치 값이 들어간다.

`trace.npz`는 배치별 `weight_norm`, `grad_norm`, `delta_norm`, `batch_loss`, `batch_lr`, `epoch_steps`, `node_ids`를 담는다. `epochs.csv`는 epoch별 학습 손실과 공식 테스트 분할 평가값을 담는다. 배치 손실은 해당 업데이트 전의 값이다.

In [ ]:
@torch.inference_mode()
def evaluate(model):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    for images, targets in test_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        logits = model(images)
        total_loss += nn.functional.cross_entropy(logits, targets, reduction="sum").item()
        total_correct += (logits.argmax(1) == targets).sum().item()
    return total_loss / len(test_data), total_correct / len(test_data)


def row_norms(tensors):
    return torch.cat([torch.linalg.vector_norm(tensor, dim=1) for tensor in tensors])


def train(seed, method):
    torch.manual_seed(seed)
    model = MLP1().to(DEVICE)
    optimizer = PaperAdam(model.parameters(), hypergrad_lr=METHODS[method])
    loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    steps_per_epoch = min(8, len(loader)) if SMOKE else len(loader)
    total_steps = EPOCHS * steps_per_epoch
    trace = np.full((total_steps + 1, 3, len(NODE_IDS)), np.nan, dtype=np.float32)
    batch_loss = np.full(total_steps + 1, np.nan)
    batch_lr = np.full(total_steps + 1, LR2)
    weights = (model.hidden1.weight, model.hidden2.weight)
    trace[0, 0] = row_norms([p.detach() for p in weights]).cpu().numpy()

    initial_loss, initial_accuracy = evaluate(model)
    metrics = [dict(epoch=0, step=0, train_loss="", test_loss=initial_loss,
                    test_accuracy=initial_accuracy, lr=LR2)]
    step = 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for images, targets in loader:
            step += 1
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(images), targets)
            loss.backward()
            with torch.no_grad():
                old_weights = tuple(parameter.clone() for parameter in weights)
                grad_norms = row_norms([parameter.grad for parameter in weights])
                optimizer.step()
                weight_norms = row_norms(weights)
                delta_norms = row_norms([parameter - old for parameter, old in zip(weights, old_weights)])
                trace[step] = torch.stack((weight_norms, grad_norms, delta_norms)).cpu().numpy()
            batch_loss[step] = loss.item()
            batch_lr[step] = optimizer.param_groups[0]["lr"]
            if step == epoch * steps_per_epoch:
                break

        test_loss, test_accuracy = evaluate(model)
        first_step = step - steps_per_epoch + 1
        metrics.append(dict(epoch=epoch, step=step, train_loss=batch_loss[first_step:step + 1].mean(),
                            test_loss=test_loss, test_accuracy=test_accuracy, lr=batch_lr[step]))

    assert step == total_steps and np.isfinite(trace[1:]).all()
    assert np.isfinite(batch_loss[1:]).all() and np.isfinite(batch_lr).all()
    run_dir = OUTPUT / f"seed-{seed}" / method
    run_dir.mkdir(parents=True)
    np.savez(run_dir / "trace.npz", weight_norm=trace[:, 0], grad_norm=trace[:, 1],
             delta_norm=trace[:, 2], batch_loss=batch_loss, batch_lr=batch_lr,
             epoch_steps=np.arange(EPOCHS + 1) * steps_per_epoch, node_ids=NODE_IDS)
    with (run_dir / "epochs.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=metrics[0])
        writer.writeheader()
        writer.writerows(metrics)
    print(f"{method:7} seed={seed} | {step:>6,} steps | accuracy={test_accuracy:.2%}", flush=True)
    return run_dir, test_accuracy


manifest = dict(batch_size=BATCH_SIZE, epochs=EPOCHS, seeds=SEEDS, methods=METHODS, lr2=LR2,
                betas=BETAS, eps=EPS, smoke=SMOKE, device=str(DEVICE), torch=torch.__version__,
                torchvision=torchvision.__version__)
(OUTPUT / "config.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
runs = {(seed, method): train(seed, method) for seed in SEEDS for method in METHODS}

for method in METHODS:
    accuracies = np.array([runs[seed, method][1] for seed in SEEDS]) * 100
    standard_error = accuracies.std(ddof=1) / np.sqrt(len(accuracies)) if len(accuracies) > 1 else None
    suffix = "" if standard_error is None else f", SE={standard_error:.2f}%p"
    print(f"{method}: 평균 정확도 {accuracies.mean():.2f}%{suffix}")
print(OUTPUT.resolve())

## 뉴런 norm의 피어슨 상관 네트워크

프로포절의 정의에 따라 관측 구간에서 두 뉴런의 norm 시계열에 대한 피어슨 상관계수 (r)을 엣지 가중치로 사용한다. 배치 그래프와 epoch 그래프를 별도로 계산한다. 상관을 계산할 수 없는 상수 시계열의 뉴런은 고립 노드로 남긴다. GraphML과 CSV에는 (|r|)이 임계값 이상인 엣지를 부호와 함께 저장한다. 임계값 적용 전 행렬도 저장하며, 정의되지 않은 상관은 NaN이다.

`BATCH_WINDOW`와 `EPOCH_WINDOW`를 파이썬 slice로 지정해 학습 초반 등 특정 구간을 분석할 수 있다. 예를 들어 첫 epoch의 초기 상태부터 마지막 배치를 포함하려면 `BATCH_WINDOW = slice(0, 1876)`을 쓴다. 각 구간에 유효한 관측값이 세 개 이상 필요하다. 연속 시점의 norm은 장기 추세를 공유할 수 있으므로, 이 엣지는 관측된 동행을 나타낸다. 후속 분석에서 `grad_norm`이나 `delta_norm`으로 바꿔 관계가 지속되는지 비교할 수 있다.

In [ ]:
ANALYSIS_SEED = SEEDS[0]
SIGNAL = "weight_norm"
EDGE_THRESHOLD = 0.7
BATCH_WINDOW = slice(None)
EPOCH_WINDOW = slice(None)


def correlation_graph(observations, node_ids):
    observations = observations[np.isfinite(observations).all(axis=1)].astype(np.float64)
    if len(observations) < 3:
        raise ValueError("상관 분석에는 유효한 관측 시점이 3개 이상 필요합니다.")

    varying = np.ptp(observations, axis=0) > 0
    centered = observations[:, varying] - observations[:, varying].mean(axis=0)
    unit_vectors = centered / np.linalg.norm(centered, axis=0)
    correlation = np.full((len(node_ids), len(node_ids)), np.nan)
    correlation[np.ix_(varying, varying)] = np.clip(unit_vectors.T @ unit_vectors, -1, 1)

    graph = nx.Graph(observations=len(observations), threshold=EDGE_THRESHOLD)
    for node in node_ids:
        layer, index = node.split(":")
        graph.add_node(node, layer=layer, unit=int(index))
    rows, cols = np.where(np.triu(np.abs(correlation) >= EDGE_THRESHOLD, k=1))
    for left, right in zip(rows, cols):
        graph.add_edge(node_ids[left], node_ids[right], weight=float(correlation[left, right]))
    return graph, correlation


fig, axes = plt.subplots(2, len(METHODS), figsize=(12, 7), squeeze=False)
for column, method in enumerate(METHODS):
    run_dir, _ = runs[ANALYSIS_SEED, method]
    with np.load(run_dir / "trace.npz") as saved:
        signal = saved[SIGNAL]
        node_ids = saved["node_ids"].tolist()
        epoch_steps = saved["epoch_steps"]
    selections = [("batch", signal[BATCH_WINDOW]),
                  ("epoch", signal[epoch_steps][EPOCH_WINDOW])]
    for row, (resolution, observations) in enumerate(selections):
        ax = axes[row, column]
        name = f"{SIGNAL}-{resolution}"
        valid_rows = np.isfinite(observations).all(axis=1).sum()
        if valid_rows < 3:
            ax.set_title(f"{method} · {resolution} (fewer than 3 observations)")
            ax.axis("off")
            continue
        graph, correlation = correlation_graph(observations, node_ids)
        np.savez(run_dir / f"{name}-correlation.npz", correlation=correlation, node_ids=node_ids)
        nx.write_graphml(graph, run_dir / f"{name}.graphml")
        with (run_dir / f"{name}-edges.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle)
            writer.writerow(["source", "target", "pearson_r"])
            writer.writerows((left, right, data["weight"]) for left, right, data in graph.edges(data=True))
        image = ax.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1, interpolation="nearest")
        ax.axvline(HIDDEN_SIZE - 0.5, color="black", linewidth=0.5)
        ax.axhline(HIDDEN_SIZE - 0.5, color="black", linewidth=0.5)
        ax.set_title(f"{method} · {resolution} | {graph.number_of_edges():,} edges")
        fig.colorbar(image, ax=ax, shrink=0.85, label="Pearson r")

fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(METHODS), figsize=(12, 3), sharey=True)
for ax, method in zip(axes, METHODS):
    with np.load(runs[ANALYSIS_SEED, method][0] / "trace.npz") as saved:
        signal = saved[SIGNAL]
        ax.plot(signal[:, 0], label="h1:000", alpha=0.65)
        ax.plot(signal[:, HIDDEN_SIZE], label="h2:000", alpha=0.65)
        if len(saved["epoch_steps"]) > 1:
            ax.scatter(saved["epoch_steps"], signal[saved["epoch_steps"], 0], s=10, label="h1 snapshots")
    ax.set(title=method, xlabel="Optimizer step", ylabel=SIGNAL)
    ax.legend()
fig.tight_layout()
plt.show()

## Adam-HD 참조 코드 라이선스

위 옵티마이저는 Baydin 등의 참조 구현을 현대 PyTorch API에 맞춰 수정했다.

```text
MIT License

Copyright (c) 2018 Atılım Güneş Baydin

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```